# 🚀 VisionAI Training with MLOps Integration

This notebook trains YOLOv11 for furniture condition detection with full MLOps support:
- **DagsHub**: Central hub for code, data, and experiments
- **MLflow**: Experiment tracking and model registry
- **DVC**: Data version control

---
## 📋 Prerequisites
1. Google Colab with GPU runtime
2. DagsHub account (https://dagshub.com)
3. Dataset prepared and versioned with DVC


## 1️⃣ Environment Setup


In [ ]:
# Install required packages
%pip install -q ultralytics>=8.3.0
%pip install -q dagshub mlflow>=2.9.0
%pip install -q dvc dvc-s3

print("✅ Packages installed successfully!")


In [ ]:
# Check GPU availability
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU Available: {gpu_name}")
    print(f"   Memory: {gpu_memory:.1f} GB")
    DEVICE = 0
else:
    print("⚠️ No GPU detected. Training will be slow.")
    DEVICE = 'cpu'


## 2️⃣ Connect to DagsHub

DagsHub provides:
- Git repository hosting
- MLflow tracking server
- DVC remote storage (S3-compatible)


In [ ]:
# DagsHub Configuration
# ⚠️ UPDATE THESE WITH YOUR DETAILS!
DAGSHUB_OWNER = "arsal6010"      # Your DagsHub username
DAGSHUB_REPO = "FYP_visionAI"    # Your repository name

# Initialize DagsHub
import dagshub
dagshub.init(repo_owner=DAGSHUB_OWNER, repo_name=DAGSHUB_REPO, mlflow=True)

print(f"✅ Connected to DagsHub: {DAGSHUB_OWNER}/{DAGSHUB_REPO}")
print(f"📊 MLflow Dashboard: https://dagshub.com/{DAGSHUB_OWNER}/{DAGSHUB_REPO}.mlflow")


In [ ]:
# Setup MLflow
import mlflow

MLFLOW_URI = f"https://dagshub.com/{DAGSHUB_OWNER}/{DAGSHUB_REPO}.mlflow"
mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment("YOLOv11-Household-Objects")

print(f"✅ MLflow configured")
print(f"   Tracking URI: {MLFLOW_URI}")


## 3️⃣ Pull Data with DVC

Instead of downloading from Google Drive, we pull versioned data from DagsHub storage.


In [ ]:
# Clone the repository (if not already)
import os

REPO_DIR = "VisionAI"

if not os.path.exists(REPO_DIR):
    !git clone https://dagshub.com/$DAGSHUB_OWNER/$DAGSHUB_REPO.git $REPO_DIR
    print(f"✅ Cloned repository to {REPO_DIR}")
else:
    print(f"📁 Repository already exists at {REPO_DIR}")

os.chdir(REPO_DIR)
print(f"📂 Working directory: {os.getcwd()}")


In [ ]:
# Configure DVC remote and pull data
!dvc remote modify origin --local auth basic
!dvc remote modify origin --local user $DAGSHUB_OWNER

# Pull the versioned dataset
!dvc pull -r origin

print("✅ Data pulled successfully!")


## 4️⃣ Training Configuration

Define hyperparameters for the training run.


In [ ]:
# Training Hyperparameters
# ⚙️ MODIFY THESE TO EXPERIMENT

CONFIG = {
    # Model
    "model": "yolo11s.pt",      # Options: yolo11n.pt, yolo11s.pt, yolo11m.pt, yolo11l.pt
    "imgsz": 640,               # Input image size
    
    # Training
    "epochs": 50,               # Number of epochs
    "batch": 16,                # Batch size (reduce if OOM)
    "lr0": 0.01,                # Initial learning rate
    "patience": 10,             # Early stopping patience
    
    # Data
    "data": "computer-vision/data/processed/data.yaml",
    
    # Output
    "project": "VisionAI_Runs",
    "name": "household_objects_v1",
}

print("📋 Training Configuration:")
for key, value in CONFIG.items():
    print(f"   {key}: {value}")


## 5️⃣ Train with MLflow Logging

Ultralytics automatically detects MLflow and logs:
- Hyperparameters
- Training metrics (loss, mAP, precision, recall)
- Model artifacts


In [ ]:
from ultralytics import YOLO

# Load model
model = YOLO(CONFIG["model"])

print(f"🤖 Loaded model: {CONFIG['model']}")
print(f"📊 MLflow will automatically log metrics to DagsHub")
print()
print("="*60)
print("🚀 STARTING TRAINING")
print("="*60)


In [ ]:
# Train the model
# Ultralytics automatically detects and logs to MLflow!

results = model.train(
    data=CONFIG["data"],
    epochs=CONFIG["epochs"],
    imgsz=CONFIG["imgsz"],
    batch=CONFIG["batch"],
    lr0=CONFIG["lr0"],
    patience=CONFIG["patience"],
    device=DEVICE,
    project=CONFIG["project"],
    name=CONFIG["name"],
    save=True,
    plots=True,
    val=True,
    verbose=True
)

print("\n" + "="*60)
print("✅ TRAINING COMPLETE!")
print("="*60)


## 6️⃣ Log Artifacts to MLflow


In [ ]:
# Paths to trained model and artifacts
RUN_DIR = f"{CONFIG['project']}/{CONFIG['name']}"
BEST_MODEL = f"{RUN_DIR}/weights/best.pt"
LAST_MODEL = f"{RUN_DIR}/weights/last.pt"

print(f"📁 Run directory: {RUN_DIR}")
print(f"🏆 Best model: {BEST_MODEL}")

# Log artifacts manually to MLflow
with mlflow.start_run(run_name=CONFIG["name"]) as run:
    # Log hyperparameters
    mlflow.log_params({
        "model": CONFIG["model"],
        "epochs": CONFIG["epochs"],
        "batch_size": CONFIG["batch"],
        "learning_rate": CONFIG["lr0"],
        "image_size": CONFIG["imgsz"],
    })
    
    # Log metrics from results
    if hasattr(results, 'results_dict'):
        metrics = {
            "mAP50": results.results_dict.get("metrics/mAP50(B)", 0),
            "mAP50-95": results.results_dict.get("metrics/mAP50-95(B)", 0),
            "precision": results.results_dict.get("metrics/precision(B)", 0),
            "recall": results.results_dict.get("metrics/recall(B)", 0),
        }
        mlflow.log_metrics(metrics)
        print(f"📊 Logged metrics: {metrics}")
    
    # Log best model
    if os.path.exists(BEST_MODEL):
        mlflow.log_artifact(BEST_MODEL, "models")
        print(f"✅ Logged best.pt to MLflow")
    
    print(f"\n🔗 View run at: {MLFLOW_URI}")


## 7️⃣ Evaluate the Model


In [ ]:
# Load the best model for evaluation
best_model = YOLO(BEST_MODEL)

# Run validation
val_results = best_model.val()

print("\n" + "="*60)
print("📊 VALIDATION RESULTS")
print("="*60)
print(f"   mAP@50:    {val_results.box.map50:.4f}")
print(f"   mAP@50-95: {val_results.box.map:.4f}")


## 8️⃣ Download Model

Download the trained model for local use or deployment.


In [ ]:
# Download the best model to your local machine
from google.colab import files

print("📥 Download best.pt to your computer:")
files.download(BEST_MODEL)


---
## 🎉 Training Complete!

### What's Next?

1. **View Experiments**: Go to your DagsHub MLflow Dashboard
2. **Compare Runs**: Try different hyperparameters and compare mAP scores
3. **Deploy Model**: Download `best.pt` and integrate with your Next.js frontend
4. **Version Data**: Add more training data and track versions with DVC

### Dashboard Links
- 📊 **MLflow**: https://dagshub.com/arsal6010/FYP_visionAI.mlflow
- 📁 **Repository**: https://dagshub.com/arsal6010/FYP_visionAI
